# START

In [1]:
# -*- coding: utf-8 -*-
"""
Kvasir-SEG Thesis Pipeline
DINOv3 + Cosine Similarity + Louvain Pruning + PVT-CASCADE

Colab-ready version of the user's UNet++ thesis implementation.
Main change:
    UNet++  ->  PVT-CASCADE

The pruning logic, train/test split, full-vs-pruned experiments,
BCE/structure-style loss, and Dice/IoU evaluation are kept aligned
with the original thesis implementation as much as possible.
"""

"\nKvasir-SEG Thesis Pipeline\nDINOv3 + Cosine Similarity + Louvain Pruning + PVT-CASCADE\n\nColab-ready version of the user's UNet++ thesis implementation.\nMain change:\n    UNet++  ->  PVT-CASCADE\n\nThe pruning logic, train/test split, full-vs-pruned experiments,\nBCE/structure-style loss, and Dice/IoU evaluation are kept aligned\nwith the original thesis implementation as much as possible.\n"

# 0. COLAB INSTALLATION

In [2]:
# ============================================================
# 0. COLAB INSTALLATION

In [3]:
# ============================================================
!pip install -q kagglehub huggingface_hub timm opencv-python scikit-image networkx python-louvain ml-collections

# Clone official CASCADE repository
import os
import sys
import subprocess
from pathlib import Path

CASCADE_DIR = "/content/CASCADE"

if not os.path.exists(CASCADE_DIR):
    !git clone -q https://github.com/SLDGroup/CASCADE.git /content/CASCADE

# IMPORTANT:
# PVT_CASCADE internally loads:
# ./pretrained_pth/pvt/pvt_v2_b2.pth
# Therefore the working directory must be the CASCADE repository.
os.chdir(CASCADE_DIR)

# Download official PVTv2-B2 ImageNet pretrained weights.
os.makedirs("/content/CASCADE/pretrained_pth/pvt", exist_ok=True)

PVT_WEIGHT = "/content/CASCADE/pretrained_pth/pvt/pvt_v2_b2.pth"

if not os.path.exists(PVT_WEIGHT):
    !wget -q -O /content/CASCADE/pretrained_pth/pvt/pvt_v2_b2.pth \
        https://github.com/whai362/PVT/releases/download/v2/pvt_v2_b2.pth

sys.path.insert(0, CASCADE_DIR)

print("CASCADE directory:", os.getcwd())
print("PVT pretrained weights:", os.path.exists(PVT_WEIGHT))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 3.9 MB/s eta 0:00:00
CASCADE directory: /content/CASCADE
PVT pretrained weights: True


# 1. IMPORTS

In [5]:
# ============================================================
import os
import time
import random
import numpy as np
import networkx as nx

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import transforms

from community import community_louvain

from transformers import AutoImageProcessor, AutoModel
import kagglehub

from lib.networks import PVT_CASCADE

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/CASCADE/lib/pvtv2.py:387: UserWarning: Overwriting pvt_v2_b0 in registry with lib.pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/CASCADE/lib/pvtv2.py:397: UserWarning: Overwriting pvt_v2_b1 in registry with lib.pvtv2.pvt_v2_b1. This is because the name being registered conflicts with an existing name. Please check if this is not expect

# 2. CONFIGURATION

In [36]:
# ============================================================
class Config:
    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------
    TRAIN_SPLIT = 0.80
    RANDOM_SEED = 42

    # --------------------------------------------------------
    # PVT-CASCADE
    # --------------------------------------------------------
    IMAGE_SIZE = 352
    BATCH_SIZE = 16
    NUM_WORKERS = 2
    PIN_MEMORY = True

    NUM_CLASSES = 1

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    EPOCHS = 5              # Change to 50/100/200 for final thesis runs
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4

    # --------------------------------------------------------
    # DINOv3 pruning
    # --------------------------------------------------------
    DINO_MODEL_NAME = "rA9del/dinov3b16"

    # Cosine similarity threshold.
    # NOTE: this is cosine similarity directly, not remapped to [0,1].
    SIMILARITY_THRESHOLD = 0.92

    # Fraction selected inside each Louvain community.
    RETENTION_RATIO = 0.2

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------
    THRESHOLD = 0.50

    # --------------------------------------------------------
    # Saving
    # --------------------------------------------------------
    OUTPUT_DIR = "/content/thesis_pvt_cascade_outputs"
    MODEL_DIR = "/content/thesis_pvt_cascade_outputs/checkpoints"

    BEST_FULL_MODEL = "best_pvt_cascade_full.pth"
    BEST_PRUNED_MODEL = "best_pvt_cascade_pruned.pth"

    FEATURE_FILE = "/content/thesis_pvt_cascade_outputs/dinov3_features.npy"
    PRUNED_INDEX_FILE = "/content/thesis_pvt_cascade_outputs/pruned_indices.npy"
    SPLIT_FILE = "/content/thesis_pvt_cascade_outputs/train_test_split.npz"


config = Config()

os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.MODEL_DIR, exist_ok=True)

# 3. REPRODUCIBILITY

In [37]:
# ============================================================
# 3. REPRODUCIBILITY

In [38]:
# ============================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducible behavior.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(config.RANDOM_SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("================================================")
print("DEVICE")
print("================================================")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE
Device: cuda
GPU: Tesla T4


# 4. DOWNLOAD KVASIR-SEG

In [39]:
# ============================================================
# 4. DOWNLOAD KVASIR-SEG

In [40]:
# ============================================================
print("\n================================================")
print("DOWNLOADING KVASIR-SEG")
print("================================================")

dataset_path = kagglehub.dataset_download("fkarimovv/kvasir-seg")

print("Dataset path:")
print(dataset_path)

DATASET_ROOT = os.path.join(dataset_path, "Kvasir-SEG")

IMAGE_DIR = os.path.join(DATASET_ROOT, "images")
MASK_DIR = os.path.join(DATASET_ROOT, "masks")

if not os.path.isdir(IMAGE_DIR):
    raise FileNotFoundError(f"Image directory not found: {IMAGE_DIR}")

if not os.path.isdir(MASK_DIR):
    raise FileNotFoundError(f"Mask directory not found: {MASK_DIR}")

print("Image directory:", IMAGE_DIR)
print("Mask directory :", MASK_DIR)


DOWNLOADING KVASIR-SEG
Using Colab cache for faster access to the 'kvasir-seg' dataset.
Dataset path:
/kaggle/input/kvasir-seg
Image directory: /kaggle/input/kvasir-seg/Kvasir-SEG/images
Mask directory : /kaggle/input/kvasir-seg/Kvasir-SEG/masks


# 5. FIND IMAGE/MASK PAIRS

In [41]:
# ============================================================
# 5. FIND IMAGE/MASK PAIRS

In [42]:
# ============================================================
def prepare_kvasir_data(image_dir, mask_dir):

    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    mask_files = sorted([
        f for f in os.listdir(mask_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    mask_set = set(mask_files)

    # Keep only images that have a mask with exactly the same filename.
    image_files = [
        f for f in image_files
        if f in mask_set
    ]

    if len(image_files) == 0:
        raise RuntimeError("No matching image/mask pairs found.")

    print(f"Matched image/mask pairs: {len(image_files)}")

    return image_files


all_images = prepare_kvasir_data(IMAGE_DIR, MASK_DIR)

Matched image/mask pairs: 1000


# 6. DATASET

In [43]:
# ============================================================
# 6. DATASET

In [44]:
# ============================================================
class KvasirSegDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir,
        image_names,
        image_size=352
    ):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_names = list(image_names)
        self.image_size = image_size

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        mask_path = os.path.join(
            self.mask_dir,
            image_name
        )

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # PVT-CASCADE is tested here at 352x352.
        image = image.resize(
            (self.image_size, self.image_size),
            Image.Resampling.BILINEAR
        )

        # IMPORTANT:
        # Never use bilinear/bicubic interpolation for a binary mask.
        mask = mask.resize(
            (self.image_size, self.image_size),
            Image.Resampling.NEAREST
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        mask = np.asarray(
            mask,
            dtype=np.uint8
        )

        mask = (mask > 127).astype(np.float32)

        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        ).float()

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0).float()

        return image, mask


base_dataset = KvasirSegDataset(
    IMAGE_DIR,
    MASK_DIR,
    all_images,
    image_size=config.IMAGE_SIZE
)

print("Total images:", len(base_dataset))

Total images: 1000


# 7. TRAIN / TEST SPLIT

In [45]:
# ============================================================
# 7. TRAIN / TEST SPLIT

In [46]:
# ============================================================
# This intentionally follows the original thesis implementation:
# 80% training / 20% testing.
#
# The test set is NEVER passed to the pruning algorithm.

indices = np.arange(len(base_dataset))

rng = np.random.default_rng(config.RANDOM_SEED)
rng.shuffle(indices)

split = int(
    config.TRAIN_SPLIT * len(base_dataset)
)

train_indices = indices[:split].tolist()
test_indices = indices[split:].tolist()

print("\n================================================")
print("TRAIN / TEST SPLIT")
print("================================================")
print("Training images:", len(train_indices))
print("Testing images :", len(test_indices))

print(
    "Overlap:",
    len(set(train_indices) & set(test_indices))
)

# Save split so future experiments can reuse exactly the same split.
np.savez(
    config.SPLIT_FILE,
    train_indices=np.asarray(train_indices),
    test_indices=np.asarray(test_indices)
)


TRAIN / TEST SPLIT
Training images: 800
Testing images : 200
Overlap: 0


# 8. DINOv3 GRAPH PRUNER

In [47]:
# ============================================================
# 8. DINOv3 GRAPH PRUNER

In [48]:
# ============================================================
class AccuracyOptimizedGraphPruner:

    def __init__(
        self,
        dataset,
        train_indices,
        device
    ):

        self.dataset = dataset
        self.train_indices = train_indices
        self.device = device

        print("\n================================================")
        print("LOADING DINOv3")
        print("================================================")

        self.processor = AutoImageProcessor.from_pretrained(
            config.DINO_MODEL_NAME
        )

        self.model = AutoModel.from_pretrained(
            config.DINO_MODEL_NAME
        ).to(device)

        self.model.eval()

        for param in self.model.parameters():
            param.requires_grad = False

        print("DINOv3 loaded and frozen.")

    @torch.no_grad()
    def extract_embeddings(self):

        embeddings = []

        print("\nExtracting DINOv3 CLS embeddings...")

        for count, idx in enumerate(self.train_indices):

            image, _ = self.dataset[idx]

            # Dataset image is [3,H,W] in [0,1].
            # Convert directly back to PIL.
            image_pil = transforms.ToPILImage()(image)

            inputs = self.processor(
                images=image_pil,
                return_tensors="pt"
            )

            inputs = {
                k: v.to(self.device)
                for k, v in inputs.items()
            }

            outputs = self.model(**inputs)

            # Global CLS token representation.
            cls_embedding = outputs.last_hidden_state[:, 0, :]

            # L2 normalization -> dot product becomes cosine similarity.
            cls_embedding = F.normalize(
                cls_embedding,
                p=2,
                dim=1
            )

            embeddings.append(
                cls_embedding.cpu().numpy().flatten()
            )

            if (count + 1) % 100 == 0:
                print(
                    f"Processed {count + 1}/"
                    f"{len(self.train_indices)}"
                )

        embeddings = np.asarray(
            embeddings,
            dtype=np.float32
        )

        print(
            "Embedding matrix:",
            embeddings.shape
        )

        return embeddings

    def prune(
        self,
        tau=0.75,
        p=0.50
    ):

        embeddings = self.extract_embeddings()

        # Save embeddings for later thesis analysis.
        np.save(
            config.FEATURE_FILE,
            embeddings
        )

        print("\n================================================")
        print("COSINE SIMILARITY GRAPH")
        print("================================================")

        # Since embeddings are L2 normalized:
        # dot product = cosine similarity.
        cosine_sim = np.matmul(
            embeddings,
            embeddings.T
        )

        cosine_sim = np.clip(
            cosine_sim,
            -1.0,
            1.0
        )

        print(
            "Similarity range:",
            float(cosine_sim.min()),
            "to",
            float(cosine_sim.max())
        )

        # Threshold graph.
        binary_edges = (
            cosine_sim >= tau
        ).astype(np.int8)

        # Remove self-loops.
        np.fill_diagonal(
            binary_edges,
            0
        )

        G = nx.from_numpy_array(
            binary_edges
        )

        print("Nodes:", G.number_of_nodes())
        print("Edges:", G.number_of_edges())

        # ----------------------------------------------------
        # Louvain
        # ----------------------------------------------------

        print("\nRunning Louvain community detection...")

        partition = community_louvain.best_partition(
            G,
            random_state=config.RANDOM_SEED
        )

        communities = {}

        for node, community_id in partition.items():

            communities.setdefault(
                community_id,
                []
            ).append(node)

        print(
            "Number of communities:",
            len(communities)
        )

        sizes = [
            len(nodes)
            for nodes in communities.values()
        ]

        print("Minimum community size :", min(sizes))
        print("Maximum community size :", max(sizes))
        print("Average community size :", np.mean(sizes))

        # ----------------------------------------------------
        # Degree-based representative selection
        # ----------------------------------------------------

        selected_local_nodes = []

        for community_id, nodes in communities.items():

            if len(nodes) <= 1:

                selected_local_nodes.append(
                    nodes[0]
                )

                continue

            # Same representative strategy as the original thesis:
            # highest graph degree inside the Louvain community.
            degrees = dict(
                G.degree(nodes)
            )

            sorted_nodes = sorted(
                nodes,
                key=lambda n: degrees[n],
                reverse=True
            )

            budget = max(
                1,
                int(np.ceil(p * len(nodes)))
            )

            selected_local_nodes.extend(
                sorted_nodes[:budget]
            )

        # Map local graph nodes back to dataset indices.
        pruned_global_indices = [
            self.train_indices[i]
            for i in selected_local_nodes
        ]

        np.save(
            config.PRUNED_INDEX_FILE,
            np.asarray(
                pruned_global_indices,
                dtype=np.int64
            )
        )

        retention = (
            len(pruned_global_indices)
            / len(self.train_indices)
        )

        print("\n================================================")
        print("PRUNING RESULT")
        print("================================================")
        print(
            "Original training samples:",
            len(self.train_indices)
        )
        print(
            "Selected training samples:",
            len(pruned_global_indices)
        )
        print(
            f"Actual retention: {retention * 100:.2f}%"
        )

        return pruned_global_indices

# 9. RUN PRUNING

In [49]:
# ============================================================
# 9. RUN PRUNING

In [50]:
# ============================================================
print("\n================================================")
print("DINOv3 + LOUVAIN DATASET PRUNING")
print("================================================")

pruner = AccuracyOptimizedGraphPruner(
    dataset=base_dataset,
    train_indices=train_indices,
    device=device
)

pruned_train_indices = pruner.prune(
    tau=config.SIMILARITY_THRESHOLD,
    p=config.RETENTION_RATIO
)


DINOv3 + LOUVAIN DATASET PRUNING

LOADING DINOv3


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

DINOv3 loaded and frozen.

Extracting DINOv3 CLS embeddings...
Processed 100/800
Processed 200/800
Processed 300/800
Processed 400/800
Processed 500/800
Processed 600/800
Processed 700/800
Processed 800/800
Embedding matrix: (800, 768)

COSINE SIMILARITY GRAPH
Similarity range: 0.2698614299297333 to 1.0
Nodes: 800
Edges: 36613

Running Louvain community detection...
Number of communities: 15
Minimum community size : 1
Maximum community size : 272
Average community size : 53.333333333333336

PRUNING RESULT
Original training samples: 800
Selected training samples: 171
Actual retention: 21.38%


# 10. DATALOADERS

In [51]:
# ============================================================
# 10. DATALOADERS

In [52]:
# ============================================================
full_train_loader = DataLoader(
    Subset(
        base_dataset,
        train_indices
    ),
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

pruned_train_loader = DataLoader(
    Subset(
        base_dataset,
        pruned_train_indices
    ),
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

test_loader = DataLoader(
    Subset(
        base_dataset,
        test_indices
    ),
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY
)

print("\nDataLoaders created.")


DataLoaders created.


# 11. PVT-CASCADE STRUCTURE LOSS

In [53]:
# ============================================================
# 11. PVT-CASCADE STRUCTURE LOSS

In [54]:
# ============================================================
class StructureLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, pred, mask):

        # Boundary-aware weighting.
        weit = 1 + 5 * torch.abs(
            F.avg_pool2d(
                mask,
                kernel_size=31,
                stride=1,
                padding=15
            ) - mask
        )

        # Weighted BCE.
        wbce = F.binary_cross_entropy_with_logits(
            pred,
            mask,
            reduction="none"
        )

        wbce = (
            (weit * wbce).sum(dim=(2, 3))
            / weit.sum(dim=(2, 3))
        )

        # Weighted IoU.
        pred_prob = torch.sigmoid(pred)

        inter = (
            pred_prob * mask * weit
        ).sum(dim=(2, 3))

        union = (
            (pred_prob + mask) * weit
        ).sum(dim=(2, 3))

        wiou = 1 - (
            (inter + 1)
            / (union - inter + 1)
        )

        return (
            wbce + wiou
        ).mean()

# 12. PVT-CASCADE MODEL FACTORY

In [55]:
# ============================================================
# 12. PVT-CASCADE MODEL FACTORY

In [56]:
# ============================================================
def create_pvt_cascade():

    print("\nCreating PVT-CASCADE...")

    model = PVT_CASCADE(
        n_class=config.NUM_CLASSES
    )

    model = model.to(device)

    return model

# 13. TRAINING FUNCTION

In [57]:
# ============================================================
# 13. TRAINING FUNCTION

In [58]:
# ============================================================
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        p1, p2, p3, p4 = model(
            images
        )

        loss1 = criterion(
            p1,
            masks
        )

        loss2 = criterion(
            p2,
            masks
        )

        loss3 = criterion(
            p3,
            masks
        )

        loss4 = criterion(
            p4,
            masks
        )

        # Deep supervision.
        loss = (
            loss1 +
            loss2 +
            loss3 +
            loss4
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return (
        total_loss /
        max(1, len(loader))
    )

# 14. EVALUATION

In [59]:
# ============================================================
# 14. EVALUATION

In [60]:
# ============================================================
@torch.no_grad()
def evaluate(
    model,
    loader,
    device
):

    model.eval()

    total_dice = 0.0
    total_iou = 0.0
    total_correct = 0
    total_pixels = 0

    n_images = 0

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        p1, p2, p3, p4 = model(
            images
        )

        # Same aggregation strategy as your PVT-CASCADE file.
        logits = (
            p1 +
            p2 +
            p3 +
            p4
        )

        probs = torch.sigmoid(
            logits
        )

        preds = (
            probs > config.THRESHOLD
        ).float()

        intersection = (
            preds * masks
        ).sum(
            dim=(1, 2, 3)
        )

        pred_area = preds.sum(
            dim=(1, 2, 3)
        )

        gt_area = masks.sum(
            dim=(1, 2, 3)
        )

        dice = (
            2 * intersection + 1e-7
        ) / (
            pred_area +
            gt_area +
            1e-7
        )

        union = (
            pred_area +
            gt_area -
            intersection
        )

        iou = (
            intersection + 1e-7
        ) / (
            union + 1e-7
        )

        total_dice += dice.sum().item()
        total_iou += iou.sum().item()

        total_correct += (
            preds == masks
        ).sum().item()

        total_pixels += masks.numel()

        n_images += images.size(0)

    avg_dice = (
        total_dice /
        max(1, n_images)
    )

    avg_iou = (
        total_iou /
        max(1, n_images)
    )

    accuracy = (
        total_correct /
        max(1, total_pixels)
    )

    return (
        avg_dice,
        avg_iou,
        accuracy
    )

# 15. EXPERIMENT RUNNER

In [61]:
# ============================================================
# 15. EXPERIMENT RUNNER

In [62]:
# ============================================================
def run_experiment(
    experiment_name,
    train_loader,
    checkpoint_name
):

    print("\n")
    print("=" * 60)
    print(f"TRAINING: {experiment_name}")
    print("=" * 60)

    model = create_pvt_cascade()

    criterion = StructureLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY
    )

    best_dice = -1.0

    history = []

    start_time = time.time()

    for epoch in range(
        config.EPOCHS
    ):

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        dice, iou, accuracy = evaluate(
            model,
            test_loader,
            device
        )

        epoch_time = (
            time.time() -
            epoch_start
        )

        history.append({
            "epoch": epoch + 1,
            "loss": train_loss,
            "dice": dice,
            "iou": iou,
            "accuracy": accuracy,
            "epoch_time": epoch_time
        })

        print(
            f"Epoch [{epoch+1}/{config.EPOCHS}] "
            f"Loss: {train_loss:.4f} | "
            f"Dice: {dice:.4f} | "
            f"IoU: {iou:.4f} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Time: {epoch_time:.1f}s"
        )

        # Keep the best model based on Dice.
        if dice > best_dice:

            best_dice = dice

            checkpoint_path = os.path.join(
                config.MODEL_DIR,
                checkpoint_name
            )

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch + 1,
                    "dice": dice,
                    "iou": iou,
                    "accuracy": accuracy,
                    "experiment": experiment_name
                },
                checkpoint_path
            )

            print(
                "  -> Saved:",
                checkpoint_path
            )

    total_time = (
        time.time() -
        start_time
    )

    # Load best checkpoint.
    checkpoint_path = os.path.join(
        config.MODEL_DIR,
        checkpoint_name
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    final_dice, final_iou, final_accuracy = evaluate(
        model,
        test_loader,
        device
    )

    return {
        "experiment": experiment_name,
        "training_time": total_time,
        "best_epoch": checkpoint["epoch"],
        "dice": final_dice,
        "iou": final_iou,
        "accuracy": final_accuracy,
        "history": history,
        "checkpoint": checkpoint_path
    }

# 16. EXPERIMENT 1 — FULL TRAINING SET

In [63]:
# ============================================================
# 16. EXPERIMENT 1 — FULL TRAINING SET

In [64]:
# ============================================================
full_result = run_experiment(
    experiment_name="PVT-CASCADE — FULL TRAINING SET",
    train_loader=full_train_loader,
    checkpoint_name=config.BEST_FULL_MODEL
)



TRAINING: PVT-CASCADE — FULL TRAINING SET

Creating PVT-CASCADE...
Epoch [1/5] Loss: 3.9832 | Dice: 0.8555 | IoU: 0.7810 | Accuracy: 0.9537 | Time: 76.8s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [2/5] Loss: 2.8308 | Dice: 0.7843 | IoU: 0.6996 | Accuracy: 0.9138 | Time: 75.9s
Epoch [3/5] Loss: 2.4347 | Dice: 0.8877 | IoU: 0.8255 | Accuracy: 0.9670 | Time: 75.9s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_full.pth
Epoch [4/5] Loss: 2.0692 | Dice: 0.8667 | IoU: 0.7951 | Accuracy: 0.9597 | Time: 76.0s
Epoch [5/5] Loss: 1.8252 | Dice: 0.8643 | IoU: 0.7939 | Accuracy: 0.9575 | Time: 75.9s


# 17. EXPERIMENT 2 — PRUNED TRAINING SET

In [65]:
# ============================================================
# 17. EXPERIMENT 2 — PRUNED TRAINING SET

In [66]:
# ============================================================
pruned_result = run_experiment(
    experiment_name="PVT-CASCADE — PRUNED TRAINING SET",
    train_loader=pruned_train_loader,
    checkpoint_name=config.BEST_PRUNED_MODEL
)



TRAINING: PVT-CASCADE — PRUNED TRAINING SET

Creating PVT-CASCADE...
Epoch [1/5] Loss: 5.0817 | Dice: 0.5597 | IoU: 0.4391 | Accuracy: 0.9030 | Time: 22.6s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [2/5] Loss: 3.8287 | Dice: 0.6293 | IoU: 0.5229 | Accuracy: 0.9045 | Time: 19.3s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [3/5] Loss: 3.4007 | Dice: 0.8076 | IoU: 0.7159 | Accuracy: 0.9395 | Time: 19.4s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [4/5] Loss: 2.9180 | Dice: 0.8117 | IoU: 0.7250 | Accuracy: 0.9408 | Time: 19.4s
  -> Saved: /content/thesis_pvt_cascade_outputs/checkpoints/best_pvt_cascade_pruned.pth
Epoch [5/5] Loss: 2.7163 | Dice: 0.8097 | IoU: 0.7223 | Accuracy: 0.9382 | Time: 19.4s


# 18. FINAL RESULTS

In [67]:
# ============================================================
# 18. FINAL RESULTS

In [68]:
# ============================================================
print("\n")
print("=" * 70)
print("FINAL THESIS RESULTS")
print("=" * 70)

print("\nConfiguration")
print("-" * 70)
print("Image size          :", config.IMAGE_SIZE)
print("Batch size          :", config.BATCH_SIZE)
print("Epochs              :", config.EPOCHS)
print("Learning rate       :", config.LEARNING_RATE)
print("Weight decay        :", config.WEIGHT_DECAY)
print("DINO model          :", config.DINO_MODEL_NAME)
print("Similarity threshold:", config.SIMILARITY_THRESHOLD)
print("Requested retention :", config.RETENTION_RATIO)

print("\nDataset")
print("-" * 70)
print("Total images        :", len(base_dataset))
print("Original train      :", len(train_indices))
print("Pruned train        :", len(pruned_train_indices))
print("Test                :", len(test_indices))

actual_retention = (
    len(pruned_train_indices) /
    len(train_indices)
)

print(
    f"Actual retention    : {actual_retention * 100:.2f}%"
)

print("\nFULL TRAINING SET")
print("-" * 70)
print(
    f"Training time : "
    f"{full_result['training_time']:.2f} sec"
)
print(
    f"Best epoch    : "
    f"{full_result['best_epoch']}"
)
print(
    f"Dice          : "
    f"{full_result['dice']:.4f}"
)
print(
    f"IoU           : "
    f"{full_result['iou']:.4f}"
)
print(
    f"Accuracy      : "
    f"{full_result['accuracy']:.4f}"
)

print("\nPRUNED TRAINING SET")
print("-" * 70)
print(
    f"Training time : "
    f"{pruned_result['training_time']:.2f} sec"
)
print(
    f"Best epoch    : "
    f"{pruned_result['best_epoch']}"
)
print(
    f"Dice          : "
    f"{pruned_result['dice']:.4f}"
)
print(
    f"IoU           : "
    f"{pruned_result['iou']:.4f}"
)
print(
    f"Accuracy      : "
    f"{pruned_result['accuracy']:.4f}"
)

print("\nCOMPARISON")
print("-" * 70)

dice_change = (
    pruned_result["dice"] -
    full_result["dice"]
)

iou_change = (
    pruned_result["iou"] -
    full_result["iou"]
)

training_speedup = (
    full_result["training_time"] /
    max(pruned_result["training_time"], 1e-8)
)

print(
    f"Dice change       : {dice_change:+.4f}"
)

print(
    f"IoU change        : {iou_change:+.4f}"
)

print(
    f"Training speedup  : {training_speedup:.2f}x"
)

print("\nSaved files")
print("-" * 70)
print(
    "DINO features :",
    config.FEATURE_FILE
)

print(
    "Pruned index  :",
    config.PRUNED_INDEX_FILE
)

print(
    "Full model    :",
    full_result["checkpoint"]
)

print(
    "Pruned model  :",
    pruned_result["checkpoint"]
)

print("\nDONE.")



FINAL THESIS RESULTS

Configuration
----------------------------------------------------------------------
Image size          : 352
Batch size          : 8
Epochs              : 5
Learning rate       : 0.0001
Weight decay        : 0.0001
DINO model          : rA9del/dinov3b16
Similarity threshold: 0.92
Requested retention : 0.2

Dataset
----------------------------------------------------------------------
Total images        : 1000
Original train      : 800
Pruned train        : 171
Test                : 200
Actual retention    : 21.38%

FULL TRAINING SET
----------------------------------------------------------------------
Training time : 381.04 sec
Best epoch    : 3
Dice          : 0.8877
IoU           : 0.8255
Accuracy      : 0.9670

PRUNED TRAINING SET
----------------------------------------------------------------------
Training time : 101.35 sec
Best epoch    : 4
Dice          : 0.8117
IoU           : 0.7250
Accuracy      : 0.9408

COMPARISON
-------------------------------

# END